# Trabajo Práctico — Series Temporales

## 02 — Limpieza y combinación de datos

**Objetivo de este notebook:** convertir los archivos anuales crudos de `data/raw/` (uno por año y por fuente) en una única tabla horaria limpia, con las tres variables alineadas en el mismo índice de tiempo (UTC), lista para el análisis exploratorio (`03_eda.ipynb`) y el modelado.

Este notebook **no descarga nada** (eso ya lo hizo `01_descarga_datos.ipynb`) y **no modela** — sólo transforma. Los archivos de `data/raw/` nunca se modifican: todo lo que se lee acá se transforma en memoria y se guarda como copia nueva en `data/processed/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from limpieza import (
    parsear_epa_aqs_rango,
    parsear_noaa_isd_rango,
    parsear_nsrdb_rango,
    combinar_series,
)

print(f"Directorio raíz del proyecto detectado: {PROJECT_ROOT}")

Directorio raíz del proyecto detectado: C:\Series_AMBA


## 1. Parseo de las tres fuentes

Cada fuente trae un archivo por año en `data/raw/<fuente>/`. `parsear_*_rango` recorre todos los años disponibles, filtra al sitio/estación fijado (ozono: Los Angeles-North Main Street, EPA AQS 06-037-1103; temperatura: estación de LAX, NOAA ISD 72295023174; radiación: mismo punto geográfico, NSRDB) y devuelve una única serie horaria en UTC.

**Nota sobre EPA AQS:** los archivos de AirData traen el país entero (varios GB sin comprimir) para filtrar un solo sitio; el parser filtra por texto antes de armar el DataFrame para que esto no tarde una eternidad, pero de todas formas descomprimir 28 años de datos nacionales lleva un par de minutos. Es normal que esta celda tarde.

In [2]:
ANIO_INICIO, ANIO_FIN = 1998, 2025

ozono = parsear_epa_aqs_rango(PROJECT_ROOT / "data/raw/epa_aqs", ANIO_INICIO, ANIO_FIN)
print(f"Ozono (EPA AQS)      : {len(ozono):,} registros | {ozono.index.min()} -> {ozono.index.max()}")

Ozono (EPA AQS)      : 231,765 registros | 1998-01-01 08:00:00+00:00 -> 2026-01-01 07:00:00+00:00


In [3]:
temperatura = parsear_noaa_isd_rango(PROJECT_ROOT / "data/raw/noaa_isd", ANIO_INICIO, ANIO_FIN)
print(f"Temperatura (NOAA ISD): {len(temperatura):,} registros | {temperatura.index.min()} -> {temperatura.index.max()}")

Temperatura (NOAA ISD): 242,408 registros | 1998-01-01 00:00:00+00:00 -> 2025-08-27 07:00:00+00:00


In [4]:
radiacion = parsear_nsrdb_rango(PROJECT_ROOT / "data/raw/nsrdb", ANIO_INICIO, ANIO_FIN)
print(f"Radiación solar (NSRDB): {len(radiacion):,} registros | {radiacion.index.min()} -> {radiacion.index.max()}")

Radiación solar (NSRDB): 245,448 registros | 1998-01-01 08:00:00+00:00 -> 2026-01-01 07:00:00+00:00


## 2. Validación de calidad por serie

Antes de combinar, se valida cada serie por separado: cantidad de horas esperadas en el rango (asumiendo una observación por hora, sin huecos), cantidad real de registros, y por lo tanto el porcentaje de faltantes. Esto separa los faltantes "de origen" (huecos reales de cada fuente) de los que van a aparecer recién al combinar (horas en las que sólo falta una de las tres variables).

In [5]:
def validar_serie(serie, nombre):
    horas_esperadas = pd.date_range(serie.index.min(), serie.index.max(), freq="h", tz="UTC")
    faltantes = len(horas_esperadas) - len(serie)
    pct = 100 * faltantes / len(horas_esperadas)
    print(f"[{nombre}]")
    print(f"  Horas esperadas en el rango : {len(horas_esperadas):,}")
    print(f"  Registros reales            : {len(serie):,}")
    print(f"  Faltantes                   : {faltantes:,} ({pct:.2f}%)")

validar_serie(ozono, "Ozono")
validar_serie(temperatura, "Temperatura")
validar_serie(radiacion, "Radiación solar")

[Ozono]
  Horas esperadas en el rango : 245,448
  Registros reales            : 231,765
  Faltantes                   : 13,683 (5.57%)
[Temperatura]
  Horas esperadas en el rango : 242,408
  Registros reales            : 242,408
  Faltantes                   : 0 (0.00%)
[Radiación solar]
  Horas esperadas en el rango : 245,448
  Registros reales            : 245,448
  Faltantes                   : 0 (0.00%)


## 3. Combinación en una sola tabla horaria

Se combinan las tres series con `pd.concat(..., axis=1)` sobre un índice horario común (join externo: no se pierde ninguna hora en la que al menos una serie tenga dato). El resultado va a tener `NaN` donde a una fuente le falte esa hora puntual.

In [6]:
combinada = combinar_series(ozono, temperatura, radiacion)
print(f"Tabla combinada: {combinada.shape[0]:,} filas x {combinada.shape[1]} columnas")
print(f"Rango: {combinada.index.min()} -> {combinada.index.max()}")
print("\nFaltantes por columna:")
print(combinada.isna().sum())

Tabla combinada: 245,456 filas x 3 columnas
Rango: 1998-01-01 00:00:00+00:00 -> 2026-01-01 07:00:00+00:00

Faltantes por columna:
ozono_ppm            13691
temperatura_c         5871
radiacion_ghi_wm2        8
dtype: int64


### Tratamiento de los faltantes

Se distingue explícitamente entre dos situaciones, para no interpolar a ciegas:

- **Huecos cortos (hasta 3 horas consecutivas):** se interpolan linealmente. Un sensor que se cae 1-2 horas y vuelve es ruido de medición, no una ausencia real de la variable física.
- **Huecos largos (más de 3 horas consecutivas):** no se inventan datos. Esas filas se descartan de la tabla final, y se registra cuántas fueron para dejarlo documentado (no es lo mismo depurar el 0,5% de las filas que el 20%).

In [7]:
MAX_HUECO_A_INTERPOLAR = 3

combinada_interpolada = combinada.interpolate(method="linear", limit=MAX_HUECO_A_INTERPOLAR)

filas_antes = len(combinada_interpolada)
combinada_final = combinada_interpolada.dropna()
filas_descartadas = filas_antes - len(combinada_final)

print(f"Filas tras interpolar huecos cortos (<= {MAX_HUECO_A_INTERPOLAR}h): {filas_antes:,}")
print(f"Filas con huecos largos descartadas                     : {filas_descartadas:,} ({100*filas_descartadas/filas_antes:.2f}%)")
print(f"Filas finales                                            : {len(combinada_final):,}")

Filas tras interpolar huecos cortos (<= 3h): 245,456
Filas con huecos largos descartadas                     : 6,378 (2.60%)
Filas finales                                            : 239,078


## 4. Guardado

Se guardan las tres series limpias por separado (mismo formato que se venía usando: un CSV por variable) y la tabla combinada final, que es la que van a usar los notebooks siguientes (`03_eda.ipynb` en adelante).

In [8]:
destino = PROJECT_ROOT / "data/processed"
destino.mkdir(parents=True, exist_ok=True)

combinada_final["ozono_ppm"].to_csv(destino / "ozono.csv", index_label="datetime_utc")
combinada_final["temperatura_c"].to_csv(destino / "temperatura.csv", index_label="datetime_utc")
combinada_final["radiacion_ghi_wm2"].to_csv(destino / "radiacion.csv", index_label="datetime_utc")
combinada_final.to_csv(destino / "serie_california.csv", index_label="datetime_utc")

print("Archivos guardados en data/processed/:")
for archivo in sorted(destino.glob("*.csv")):
    print(f"  - {archivo.name} ({archivo.stat().st_size / 1024:.1f} KB)")

Archivos guardados en data/processed/:
  - ozono.csv (7706.7 KB)
  - radiacion.csv (7456.1 KB)
  - serie_california.csv (10194.2 KB)
  - temperatura.csv (7639.1 KB)


In [9]:
print("Resumen final")
print("=" * 40)
print(f"Registros                : {len(combinada_final):,}")
print(f"Período                  : {combinada_final.index.min()} -> {combinada_final.index.max()}")
print(f"Columnas                 : {list(combinada_final.columns)}")
print(f"\n¿Supera los 100.000 registros por serie? {'Sí' if len(combinada_final) > 100_000 else 'No'}")
print("\nProyecto listo para continuar con 03_eda.ipynb")

Resumen final
Registros                : 239,078
Período                  : 1998-01-01 08:00:00+00:00 -> 2025-08-27 09:00:00+00:00
Columnas                 : ['ozono_ppm', 'temperatura_c', 'radiacion_ghi_wm2']

¿Supera los 100.000 registros por serie? Sí

Proyecto listo para continuar con 03_eda.ipynb
